# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [34]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [35]:
df_cleaned = pd.read_csv("zillow_cleaned.csv")

# ensure we have no NAs for models to work
df_cleaned = df_cleaned.dropna()

df_cleaned = df_cleaned.drop(columns=['propertyzoningdesc', 'propertycountylandusecode']) # dropping propertyzoningdesc and propertycountylandusecode because they have too many unique string values. It could be encoded, but we decided against that.
df_cleaned = df_cleaned.drop(columns=['bathroomcnt', 'fullbathcnt', 'finishedsquarefeet12']) # dropping these features since they are redundant to other bathroom/sqft features with near perfect positive correlations 
random_state = 42

In [36]:
X = df_cleaned.drop(columns=['taxvaluedollarcnt'])
y = df_cleaned['taxvaluedollarcnt']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [38]:
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

#### Ridge Regression - Baseline

In [39]:
# Ridge Regression
ridge_model = Ridge(random_state=random_state)
ridge_scores = -cross_val_score(ridge_model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')

ridge_mean_mae = np.mean(ridge_scores)
ridge_std_mae = np.std(ridge_scores)

print(f"Mean CV MAE Score: ${ridge_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${ridge_std_mae:,.2f}")


Mean CV MAE Score: $236,831.33
STD CV MAE Score: $47,231.01


#### Random Forest - Baseline

In [40]:
# Random Forest
forest_model = RandomForestRegressor(random_state=random_state)
forest_scores = -cross_val_score(forest_model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')

forest_mean_mae = np.mean(forest_scores)
forest_std_mae = np.std(forest_scores)

print(f"Mean CV MAE Score: ${forest_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${forest_std_mae:,.2f}")

Mean CV MAE Score: $184,979.41
STD CV MAE Score: $11,173.44


#### Gradient Boosting Trees - Baseline

In [41]:
# Gradient Boosting Trees
boosting_model = GradientBoostingRegressor(random_state=random_state)
boosting_scores = -cross_val_score(boosting_model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')

boosting_mean_mae = np.mean(boosting_scores)
boosting_std_mae = np.std(boosting_scores)

print(f"Mean CV MAE Score: ${boosting_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${boosting_std_mae:,.2f}")

Mean CV MAE Score: $185,165.87
STD CV MAE Score: $10,869.67


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

> Your text here

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [42]:
# New Features

df_engineered = df_cleaned.copy()

# Luxury Square Feet
df_engineered['luxury_sqft'] = (df_engineered['hashottuborspa'] + df_engineered['poolcnt'] + df_engineered['fireplacecnt']) * df_engineered['calculatedfinishedsquarefeet']

# Bedrooms Per Bathroom
df_engineered['bedrooms_per_bathroom'] = df_engineered['bedroomcnt'] / df_engineered['calculatedbathnbr']

# Living Space to Lot Ratio
df_engineered['living_space_to_lot_ratio'] = df_engineered['calculatedfinishedsquarefeet'] / df_engineered['lotsizesquarefeet']

# Average Room Sauare Feet
df_engineered['avg_room_sqft'] = df_engineered['calculatedfinishedsquarefeet'] / df_engineered['roomcnt']

# Square Feet Per Story
df_engineered['sqft_per_story'] = df_engineered['calculatedfinishedsquarefeet'] / df_engineered['numberofstories']

# Mansion
df_engineered['is_mansion'] = ((df_engineered['unitcnt'] == 1) & (df_engineered['bedroomcnt'] >= 5) & (df_engineered['calculatedfinishedsquarefeet'] >= 5000) &
((df_engineered['hashottuborspa'] == 1) | (df_engineered['poolcnt'] > 0) | (df_engineered['fireplacecnt'] > 0))).astype(int)

In [43]:
# Log-Transformed Features (for skewed features)

df_engineered['calculatedbathnbr_log'] = np.log1p(df_engineered['calculatedbathnbr'])

df_engineered['garagecarcnt_log'] = np.log1p(df_engineered['garagecarcnt'])

df_engineered['lotsizesquarefeet_log'] = np.log1p(df_engineered['lotsizesquarefeet'])

In [44]:
# Polynomial-Transformed Features (to capture non-linear feature interactions)

df_engineered['calculatedfinishedsquarefeet_x_calculatedbathnbr'] = df_engineered['calculatedfinishedsquarefeet'] * df_engineered['calculatedbathnbr']

df_engineered['fireplacecnt_x_garagetotalsqft'] = df_engineered['fireplacecnt'] * df_engineered['garagetotalsqft']

df_engineered['calculatedbathnbr_x_numberofstories'] = df_engineered['calculatedbathnbr'] * df_engineered['numberofstories']

In [45]:
#Re-run train test split and standard scaler with the new engineered features added

X_eng = df_engineered.drop(columns=['taxvaluedollarcnt'])
y_eng = df_engineered['taxvaluedollarcnt']

X_eng_train, X_eng_test, y_eng_train, y_eng_test = train_test_split(X_eng, y_eng, test_size=0.2, random_state=random_state)

scaler = StandardScaler()
X_eng_train_scaled = scaler.fit_transform(X_eng_train)

In [46]:
# Initiate Cross-Validation
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

#### Ridge Regression - All Engineered Features

In [ ]:
# Ridge Regression with all Engineered Features Added
ridge_model = Ridge(random_state=random_state)
ridge_scores = -cross_val_score(ridge_model, X_eng_train_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

ridge_mean_mae = np.mean(ridge_scores)
ridge_std_mae = np.std(ridge_scores)

print(f"Mean CV MAE Score: ${ridge_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${ridge_std_mae:,.2f}")

Mean CV MAE Score: $236,576.45
STD CV MAE Score: $48,937.99


#### Ridge Regression - New Features

In [53]:
X_new = X_eng_train.drop(['calculatedbathnbr_log', 'garagecarcnt_log', 'lotsizesquarefeet_log',
                                    'calculatedfinishedsquarefeet_x_calculatedbathnbr', 'fireplacecnt_x_garagetotalsqft', 'calculatedbathnbr_x_numberofstories'], axis = 1)

scaler = StandardScaler()
X_new_scaled = scaler.fit_transform(X_new)

In [ ]:
# Ridge Regression with New Features Added
ridge_model = Ridge(random_state=random_state)
ridge_scores = -cross_val_score(ridge_model, X_new_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

ridge_mean_mae = np.mean(ridge_scores)
ridge_std_mae = np.std(ridge_scores)

print(f"Mean CV MAE Score: ${ridge_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${ridge_std_mae:,.2f}")

Mean CV MAE Score: $235,780.13
STD CV MAE Score: $48,988.48


#### Ridge Regression - Log Transformed Features

In [56]:
X_log = X_eng_train.drop(['calculatedfinishedsquarefeet_x_calculatedbathnbr', 'fireplacecnt_x_garagetotalsqft', 'calculatedbathnbr_x_numberofstories',
                            'luxury_sqft', 'bedrooms_per_bathroom', 'living_space_to_lot_ratio',
                            'avg_room_sqft', 'sqft_per_story', 'is_mansion'], axis = 1)

scaler = StandardScaler()
X_log_scaled = scaler.fit_transform(X_log)

In [ ]:
# Ridge Regression with Log Features Added
ridge_model = Ridge(random_state=random_state)
ridge_scores = -cross_val_score(ridge_model, X_log_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

ridge_mean_mae = np.mean(ridge_scores)
ridge_std_mae = np.std(ridge_scores)

print(f"Mean CV MAE Score: ${ridge_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${ridge_std_mae:,.2f}")

Mean CV MAE Score: $236,877.28
STD CV MAE Score: $48,328.82


#### Ridge Regression - Polynomial Transformed Features

In [58]:
X_poly = X_eng_train.drop(['calculatedbathnbr_log', 'garagecarcnt_log', 'lotsizesquarefeet_log',
                            'luxury_sqft', 'bedrooms_per_bathroom', 'living_space_to_lot_ratio',
                            'avg_room_sqft', 'sqft_per_story', 'is_mansion'], axis = 1)

scaler = StandardScaler()
X_poly_scaled = scaler.fit_transform(X_poly)

In [60]:
# Ridge Regression with Polynomial Features Added
ridge_model = Ridge(random_state=random_state)
ridge_scores = -cross_val_score(ridge_model, X_poly_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

ridge_mean_mae = np.mean(ridge_scores)
ridge_std_mae = np.std(ridge_scores)

print(f"Mean CV MAE Score: ${ridge_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${ridge_std_mae:,.2f}")

Mean CV MAE Score: $235,017.13
STD CV MAE Score: $47,016.58


#### Random Forest - All Engineered Features

In [ ]:
# Random Forest with all Engineered Features Added
forest_model = RandomForestRegressor(random_state=random_state)
forest_scores = -cross_val_score(forest_model, X_eng_train_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

forest_mean_mae = np.mean(forest_scores)
forest_std_mae = np.std(forest_scores)

print(f"Mean CV MAE Score: ${forest_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${forest_std_mae:,.2f}")

Mean CV MAE Score: $186,078.84
STD CV MAE Score: $11,011.35


#### Random Forest - New Features

In [61]:
# Random Forest with New Features Added
forest_model = RandomForestRegressor(random_state=random_state)
forest_scores = -cross_val_score(forest_model, X_new_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

forest_mean_mae = np.mean(forest_scores)
forest_std_mae = np.std(forest_scores)

print(f"Mean CV MAE Score: ${forest_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${forest_std_mae:,.2f}")

Mean CV MAE Score: $185,983.46
STD CV MAE Score: $11,065.30


#### Random Forest - Log Transformed Features

In [62]:
# Random Forest with Log Features Added
forest_model = RandomForestRegressor(random_state=random_state)
forest_scores = -cross_val_score(forest_model, X_log_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

forest_mean_mae = np.mean(forest_scores)
forest_std_mae = np.std(forest_scores)

print(f"Mean CV MAE Score: ${forest_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${forest_std_mae:,.2f}")

Mean CV MAE Score: $184,707.13
STD CV MAE Score: $10,987.43


#### Random Forest - Polynomial Transformed Features

In [63]:
# Random Forest with Polynomial Features Added
forest_model = RandomForestRegressor(random_state=random_state)
forest_scores = -cross_val_score(forest_model, X_poly_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

forest_mean_mae = np.mean(forest_scores)
forest_std_mae = np.std(forest_scores)

print(f"Mean CV MAE Score: ${forest_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${forest_std_mae:,.2f}")

Mean CV MAE Score: $184,918.62
STD CV MAE Score: $10,573.03


#### Gradient Boosting Trees - All Engineered Features

In [ ]:
# Gradient Boosting Trees with all Engineered Features Added
boosting_model = GradientBoostingRegressor(random_state=random_state)
boosting_scores = -cross_val_score(boosting_model, X_eng_train_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

boosting_mean_mae = np.mean(boosting_scores)
boosting_std_mae = np.std(boosting_scores)

print(f"Mean CV MAE Score: ${boosting_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${boosting_std_mae:,.2f}")

Mean CV MAE Score: $184,738.25
STD CV MAE Score: $11,188.97


#### Gradient Boosting Trees - New Features

In [64]:
# Gradient Boosting Trees with New Features Added
boosting_model = GradientBoostingRegressor(random_state=random_state)
boosting_scores = -cross_val_score(boosting_model, X_new_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

boosting_mean_mae = np.mean(boosting_scores)
boosting_std_mae = np.std(boosting_scores)

print(f"Mean CV MAE Score: ${boosting_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${boosting_std_mae:,.2f}")

Mean CV MAE Score: $185,263.67
STD CV MAE Score: $10,962.21


#### Gradient Boosting Trees - Log Transformed Features

In [65]:
# Gradient Boosting Trees with Log Features Added
boosting_model = GradientBoostingRegressor(random_state=random_state)
boosting_scores = -cross_val_score(boosting_model, X_log_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

boosting_mean_mae = np.mean(boosting_scores)
boosting_std_mae = np.std(boosting_scores)

print(f"Mean CV MAE Score: ${boosting_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${boosting_std_mae:,.2f}")

Mean CV MAE Score: $185,100.91
STD CV MAE Score: $10,976.69


#### Gradient Boosting Trees - Polynomial Transformed Features

In [66]:
# Gradient Boosting Trees with Polynomial Features Added
boosting_model = GradientBoostingRegressor(random_state=random_state)
boosting_scores = -cross_val_score(boosting_model, X_poly_scaled, y_eng_train, cv=cv, scoring='neg_mean_absolute_error')

boosting_mean_mae = np.mean(boosting_scores)
boosting_std_mae = np.std(boosting_scores)

print(f"Mean CV MAE Score: ${boosting_mean_mae:,.2f}")
print(f"STD CV MAE Score: ${boosting_std_mae:,.2f}")

Mean CV MAE Score: $184,663.68
STD CV MAE Score: $11,212.66


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




> Your text here

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [7]:
# Add as many cells as you need


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [8]:
# Add as many cells as you need


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [9]:
# Add as many cells as you need


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

> Your text here